In [174]:
from pymongo import MongoClient
import requests
import json
import pandas as pd
from thefuzz import process
import re

In [175]:
res = requests.get("http://110.164.203.137:9919/anime")
data = res.json() 
anime = pd.DataFrame(data)
comment = pd.read_json('anime_2025_thread_comments_new.json')

In [176]:
# 1. ฟังก์ชันล้างคำส่วนเกิน (Cleaning) เพื่อให้ Match ง่ายขึ้น
def clean_anime_name(text):
    if pd.isna(text): return ""
    text = text.lower()
    # ลบคำที่มักจะทำให้การ Match เพี้ยน
    text = re.sub(r'season \d+|episode of|the animation|:', '', text)
    # ลบช่องว่างส่วนเกิน
    return " ".join(text.split())

# 2. เตรียมข้อมูล (สมมติว่าคุณมี comment และ anime อยู่แล้ว)
# สร้างคอลัมน์ชั่วคราวสำหรับเทียบ (Cleaned Name)
comment['clean_name'] = comment['anime'].apply(clean_anime_name)
anime['clean_title'] = anime['title'].apply(clean_anime_name)


In [177]:
display(anime.sample(5))

,mal_id,url,season,year,trailer,title,title_english,type,source,rating,score,genres,clean_title
216,60931,https://myanimelist.net/anime/60931/Shentan_Xi...,winter,2025,"{'youtube_id': None, 'url': None, 'embed_url':...",Shentan Xiao Wu,NaN,ONA,Original,PG - Children,NaN,"[{'mal_id': 2, 'type': 'anime', 'name': 'Adven...",shentan xiao wu
85,61134,https://myanimelist.net/anime/61134/Kimi_ga_Sh...,winter,2025,"{'youtube_id': None, 'url': None, 'embed_url':...",Kimi ga Shinu made Koi wo Shitai Pilot,I Want to Love You Till Your Dying Day Pilot Film,PV,Manga,PG-13 - Teens 13 or older,7.65,"[{'mal_id': 1, 'type': 'anime', 'name': 'Actio...",kimi ga shinu made koi wo shitai pilot
66,60521,https://myanimelist.net/anime/60521/Twins_Hina...,winter,2025,"{'youtube_id': None, 'url': None, 'embed_url':...",Twins Hinahima,NaN,TV Special,Original,PG-13 - Teens 13 or older,5.00,"[{'mal_id': 24, 'type': 'anime', 'name': 'Sci-...",twins hinahima
580,62352,https://myanimelist.net/anime/62352/Chainsaw_M...,summer,2025,"{'youtube_id': None, 'url': None, 'embed_url':...",Chainsaw Man Recap,Chainsaw Man: The Compilation,ONA,Manga,R - 17+ (violence & profanity),7.79,"[{'mal_id': 1, 'type': 'anime', 'name': 'Actio...",chainsaw man recap
96,61179,https://myanimelist.net/anime/61179/Kimetsu_no...,winter,2025,"{'youtube_id': None, 'url': None, 'embed_url':...",Kimetsu no Yaiba x MLB,Demon Slayer: Kimetsu no Yaiba x MLB,ONA,Other,PG-13 - Teens 13 or older,6.29,"[{'mal_id': 30, 'type': 'anime', 'name': 'Spor...",kimetsu no yaiba x mlb


In [178]:
# 3. สร้าง Dictionary เพื่อเก็บผลการ Match (กันการคำนวณซ้ำเพื่อความเร็ว)
unique_comments = comment['clean_name'].unique()
choices = anime['clean_title'].tolist()
match_dict = {}

print("กำลังวิเคราะห์ความคล้ายของชื่อเรื่อง... (Fuzzy Matching)")

for name in unique_comments:
    # หาชื่อที่ใกล้เคียงที่สุด (Threshold 80% คือกำลังดี)
    best_match, score = process.extractOne(name, choices)
    if score >= 80:
        # หา Index ของชื่อที่ Match ติดในตาราง anime เพื่อดึงชื่อ 'ต้นฉบับ' มา
        original_title = anime.iloc[choices.index(best_match)]['title']
        match_dict[name] = original_title
    else:
        match_dict[name] = None


กำลังวิเคราะห์ความคล้ายของชื่อเรื่อง... (Fuzzy Matching)


In [179]:
# 4. นำผลลัพธ์ที่ Match ได้ กลับไปใส่ในตาราง comment
comment['matched_title'] = comment['clean_name'].map(match_dict)

# 5. Merge ข้อมูลเข้าด้วยกัน
df_merged = pd.merge(comment, anime, left_on='matched_title', right_on='title', how='left')

# แสดงผลลัพธ์ที่ Match ติด
matched_count = df_merged['title'].notna().sum()
print(f"--- สรุปผล ---")
print(f"จับคู่สำเร็จ: {matched_count} แถว")
print(df_merged[['anime', 'title']].sample(10)) # ดูตัวอย่างการจับคู่

--- สรุปผล ---
จับคู่สำเร็จ: 8256 แถว
                                                 anime  \
4216                 Rock wa Lady no Tashinami Deshite   
681                                 Tsuyokute New Saga   
7187                 Puniru wa Kawaii Slime 2nd Season   
3204                           Osomatsu-san 4th Season   
5288                      Shoushimin Series 2nd Season   
1175                                           Lazarus   
5399                         Kaoru Hana wa Rin to Saku   
6708                     Kidou Senshi Gundam GQuuuuuuX   
2992  Kuroiwa Medaka ni Watashi no Kawaii ga Tsuujinai   
6227                              SAKAMOTO DAYS Part 2   

                                                 title  
4216                 Rock wa Lady no Tashinami deshite  
681                                 Tsuyokute New Saga  
7187                 Puniru wa Kawaii Slime 2nd Season  
3204                           Osomatsu-san 4th Season  
5288                      Shoushimin S

In [180]:
cols_to_drop = ['clean_name', 'clean_title', 'matched_title']

df_final = df_merged.drop(columns=[c for c in cols_to_drop if c in df_merged.columns])

# 2. ตรวจสอบดูว่าเหลือค่าว่าง (NaN) อีกกี่แถว
print("จำนวนค่าว่างคงเหลือในคอลัมน์ title:", df_final['title'].isna().sum())

จำนวนค่าว่างคงเหลือในคอลัมน์ title: 26


In [181]:
# กรองเฉพาะแถวที่คอลัมน์ 'title' (จากฝั่งขวา) เป็นค่าว่าง
missing_data = df_merged[df_merged['title'].isnull()]

# แสดงรายชื่ออนิเมะที่ไม่ซ้ำกันที่ Merge ไม่ติด
print("รายชื่ออนิเมะที่ยังว่างอยู่ (หาคู่ไม่เจอ):")
print(missing_data['anime'].unique())

รายชื่ออนิเมะที่ยังว่างอยู่ (หาคู่ไม่เจอ):
<StringArray>
['Onmyo Kaiten Re:verse', 'IRIS OUT']
Length: 2, dtype: str


In [182]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 8282 entries, 0 to 8281
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   anime          8282 non-null   str    
 1   anime_id       8282 non-null   int64  
 2   thread_id      8282 non-null   int64  
 3   thread_title   8282 non-null   str    
 4   user           8282 non-null   str    
 5   text           8282 non-null   str    
 6   clean_name     8282 non-null   str    
 7   matched_title  8256 non-null   str    
 8   mal_id         8256 non-null   float64
 9   url            8256 non-null   str    
 10  season         8256 non-null   str    
 11  year           8256 non-null   float64
 12  trailer        8256 non-null   object 
 13  title          8256 non-null   str    
 14  title_english  7808 non-null   str    
 15  type           8256 non-null   str    
 16  source         8256 non-null   str    
 17  rating         8256 non-null   str    
 18  score          8251

In [183]:
df_merged.loc[df_merged['anime'] == "Onmyo Kaiten Re:verse", ['anime']] = "Onmyou Kaiten Re:Birth"


target_id = 48514  # ตัวอย่าง ID
df_merged.loc[df_merged['anime'] == "Onmyou Kaiten Re:Birth", 'mal_id'] = 48514


# 1. ลบแถวที่ชื่อ 'IRIS OUT' ออกจาก df_final
# เราใช้เงื่อนไขหา Index ของแถวนั้นแล้วสั่ง drop
df_final = df_merged.drop(df_merged[df_final['anime'] == 'IRIS OUT'].index)
# display(df_final)
# 2. เลือกเฉพาะคอลัมน์ที่ต้องการ (สมมติว่าเอา anime, mal_id, และ score)
# เปลี่ยนชื่อในลิสต์ด้านล่างตามคอลัมน์ที่คุณอยากได้จริงได้เลยครับ
selected_columns = ["anime","thread_id","thread_title","user","text","mal_id"]
df_final = df_final[selected_columns]
df_final['mal_id'] = df_final['mal_id'].astype(int)


display(df_final)

,anime,thread_id,thread_title,user,text,mal_id
0,Tomodachi no Imouto ga Ore ni dake Uzai,38418,Art similarities between this anime/ln/manga a...,DontMindMe1,I can see where you're coming from and they do...,47158
1,Tomodachi no Imouto ga Ore ni dake Uzai,38418,Art similarities between this anime/ln/manga a...,DibDibs,I won't be surprised if I found out ImoUza too...,47158
2,Tomodachi no Imouto ga Ore ni dake Uzai,38418,Art similarities between this anime/ln/manga a...,Jesse,I (personally) don't think the two Iroha's loo...,47158
3,Tomodachi no Imouto ga Ore ni dake Uzai,38418,Art similarities between this anime/ln/manga a...,catlyn,maybe,47158
4,Tomodachi no Imouto ga Ore ni dake Uzai,70253,"This has been announced 3 years ago, so any ne...",ChomikKamikadze,The newest info on the anime's official Twitte...,47158
...,...,...,...,...,...,...
8270,Jujutsu Kaisen: Shibuya Jihen Tokubetsu Henshu...,86514,I thought recap movie with no new contents wer...,Removed,[Removed],62392
8271,Jujutsu Kaisen: Shibuya Jihen Tokubetsu Henshu...,86514,I thought recap movie with no new contents wer...,NepNep,I think that's for Recuts (as per the submissi...,62392
8272,Jujutsu Kaisen: Shibuya Jihen Tokubetsu Henshu...,86514,I thought recap movie with no new contents wer...,KillerTacos,This movie was such a waste of time man. The f...,62392
8273,Jujutsu Kaisen: Shibuya Jihen Tokubetsu Henshu...,86514,I thought recap movie with no new contents wer...,AshMesa,"I saw it, there was Season 3 Episode 1-2 conte...",62392


In [188]:
# บันทึก df_final เป็นไฟล์ .json
df_final.to_json('clean_comment_anilst.json', orient='records', force_ascii=False, indent=4)

print("บันทึกไฟล์สำเร็จแล้ว!")

บันทึกไฟล์สำเร็จแล้ว!


In [187]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 8275 entries, 0 to 8274
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   anime         8275 non-null   str  
 1   thread_id     8275 non-null   int64
 2   thread_title  8275 non-null   str  
 3   user          8275 non-null   str  
 4   text          8275 non-null   str  
 5   mal_id        8275 non-null   int64
dtypes: int64(2), str(4)
memory usage: 388.0 KB


In [186]:
# แสดงทุกแถวที่มีค่าว่างอย่างน้อย 1 คอลัมน์
display(df_final[df_final.isnull().any(axis=1)])

# หรือแสดงเฉพาะแถวที่ mal_id เป็นค่าว่าง
display(df_final[df_final['mal_id'].isnull()])

,anime,thread_id,thread_title,user,text,mal_id


,anime,thread_id,thread_title,user,text,mal_id
